# Kareerly Baseline Hybrid Matching Model v1 — Colab Evaluation

This notebook evaluates the **current Kareerly baseline model** before any competency-aware upgrades.

**Scope of this notebook**
- Uses existing backend parsing, extraction, matching, gap, recommendation, and validation behavior.
- Computes baseline evaluation metrics for thesis reporting.

**Important**
- No backend formulas or logic are modified here.
- This is a reproducible evaluation workflow only.


## 1) Install Required Packages

Installs only evaluation dependencies from the import checklist.


In [ ]:
import importlib.util
import subprocess
import sys

required_packages = [
    "pandas",
    "numpy",
    "scikit-learn",
    "scipy",
    "joblib",
    "pypdf",
    "python-docx",
]

missing = []
for pkg in required_packages:
    module_name = {
        "scikit-learn": "sklearn",
        "python-docx": "docx",
    }.get(pkg, pkg)
    if importlib.util.find_spec(module_name) is None:
        missing.append(pkg)

if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
else:
    print("All required evaluation packages are already installed.")


## 2) Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3) Setup Paths and Imports

Assumes project is located at:
`/content/drive/MyDrive/kareerly_system/`


In [ ]:
import sys
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/kareerly_system')
BACKEND_DIR = PROJECT_DIR / 'backend'
APP_DIR = BACKEND_DIR / 'app'
DATA_DIR = BACKEND_DIR / 'data'
MODELS_DIR = BACKEND_DIR / 'baseline_models'
EVALUATION_DIR = BACKEND_DIR / 'evaluation'
SAMPLE_RESUMES_DIR = EVALUATION_DIR / 'sample_resumes'
OUTPUTS_DIR = EVALUATION_DIR / 'outputs'

for p in [BACKEND_DIR, APP_DIR, EVALUATION_DIR]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_RESUMES_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_DIR =', PROJECT_DIR)
print('BACKEND_DIR =', BACKEND_DIR)
print('APP_DIR =', APP_DIR)
print('DATA_DIR =', DATA_DIR)
print('MODELS_DIR =', MODELS_DIR)
print('EVALUATION_DIR =', EVALUATION_DIR)
print('SAMPLE_RESUMES_DIR =', SAMPLE_RESUMES_DIR)
print('OUTPUTS_DIR =', OUTPUTS_DIR)


## 4) Run Import Test Script

This reuses the existing readiness checker:
`backend/evaluation/colab_import_test.py`


In [ ]:
import subprocess
import sys

import_test_path = EVALUATION_DIR / 'colab_import_test.py'
if not import_test_path.exists():
    print(f'WARNING: Import test script not found at {import_test_path}')
else:
    cmd = [sys.executable, str(import_test_path), '--project-dir', str(PROJECT_DIR)]
    print('Running:', ' '.join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print('WARNING: Import test reported issues. Continue after reviewing warnings/errors above.')
    if result.stderr.strip():
        print('--- STDERR ---')
        print(result.stderr)


## 5) Load Backend Modules and Common Helpers

In [ ]:
import json
import math
from collections import defaultdict

import numpy as np
import pandas as pd

resume_analysis_cache = {}
matching_cache = {}
warnings_log = []


def warn(msg: str):
    warnings_log.append(msg)
    print(f'WARNING: {msg}')


# Try importing baseline modules.
backend_ready = True
try:
    from app.services.skill_extractor import SkillExtractor
    from app.services.resume_analyzer import analyze_resume_file
    from app.services.job_matcher import JobMatcher
    from app.services.learning_recommender import recommend_learning_resources
except Exception as exc:
    backend_ready = False
    warn(f'Backend import failed: {exc}')


# Certificate validator: prefer backend function; fallback mirrors baseline logic.
certificate_validator_source = 'backend'
try:
    from app.main import _is_valid_certificate_pdf as validate_certificate_pdf
except Exception as exc:
    certificate_validator_source = 'fallback-notebook'
    warn(f'Could not import app.main certificate validator ({exc}). Using notebook fallback with same baseline checks.')

    def validate_certificate_pdf(filename: str, file_bytes: bytes):
        lower_name = filename.lower().strip()
        if not lower_name.endswith('.pdf'):
            return False, 'Only PDF certificate files are accepted.'
        if len(file_bytes) > 5 * 1024 * 1024:
            return False, 'Certificate file must be 5MB or smaller.'
        if not file_bytes.startswith(b'%PDF'):
            return False, 'Invalid PDF signature detected.'
        decoded = file_bytes.decode('latin1', errors='ignore')
        if '/Type /Page' not in decoded:
            return False, 'The uploaded file does not appear to be a valid PDF document.'
        has_certificate_keyword = any(
            keyword in lower_name
            for keyword in [
                'certificate', 'cert', 'tesda', 'diploma', 'credential', 'license', 'licence', 'training'
            ]
        )
        has_document_keyword = any(
            keyword in decoded.lower()
            for keyword in [
                'certificate', 'certification', 'tesda', 'diploma', 'awarded', 'issued',
                'completion', 'credential', 'license', 'licence'
            ]
        )
        if not has_certificate_keyword and not has_document_keyword:
            return False, 'No certificate markers were detected in filename or document content.'
        return True, 'Certificate file passed baseline validation checks.'

print('backend_ready =', backend_ready)
print('certificate_validator_source =', certificate_validator_source)


## 6) Load Evaluation CSV Templates

Loads (or creates if missing):
- `expected_skills.csv`
- `expected_job_matches.csv`
- `expected_skill_gaps.csv`
- `expected_course_recommendations.csv`
- `file_validation_cases.csv`

If a file is empty, the notebook warns and continues.


In [ ]:
template_specs = {
    'expected_skills.csv': ['resume_id', 'skill_id', 'skill_name'],
    'expected_job_matches.csv': ['resume_id', 'job_id', 'expected_rank', 'relevance'],
    'expected_skill_gaps.csv': ['resume_id', 'job_id', 'skill_id'],
    'expected_course_recommendations.csv': ['resume_id', 'resource_id', 'skill_id'],
    'file_validation_cases.csv': ['file_name', 'expected_valid', 'notes'],
}


def load_or_create_csv(path: Path, columns: list[str]):
    if not path.exists():
        pd.DataFrame(columns=columns).to_csv(path, index=False)
        warn(f'Created missing template: {path.name}')

    try:
        df = pd.read_csv(path, dtype=str, keep_default_na=False)
    except Exception as exc:
        warn(f'Could not read {path.name}: {exc}. Using empty DataFrame.')
        return pd.DataFrame(columns=columns)

    df.columns = [str(c).strip() for c in df.columns]

    for c in columns:
        if c not in df.columns:
            df[c] = ''

    if df.empty:
        warn(f'{path.name} is empty. Metrics depending on this file will be skipped.')

    return df


expected_data = {}
for filename, cols in template_specs.items():
    expected_data[filename] = load_or_create_csv(EVALUATION_DIR / filename, cols)

expected_skills_df = expected_data['expected_skills.csv']
expected_jobs_df = expected_data['expected_job_matches.csv']
expected_gaps_df = expected_data['expected_skill_gaps.csv']
expected_courses_df = expected_data['expected_course_recommendations.csv']
file_validation_cases_df = expected_data['file_validation_cases.csv']

for name, df in expected_data.items():
    print(f'{name}: {len(df)} rows')


## 7) Prepare Resume File List and Utility Functions

In [ ]:
resume_files = sorted(
    [p for p in SAMPLE_RESUMES_DIR.glob('*') if p.is_file() and p.suffix.lower() in {'.pdf', '.docx'}]
)

if not resume_files:
    warn('No PDF/DOCX files found in sample_resumes/. Add files to continue evaluations.')

print('Sample resumes found:', len(resume_files))
for p in resume_files[:20]:
    print('-', p.name)


def normalize_text(value):
    return str(value).strip().lower()


def to_bool(value):
    text = str(value).strip().lower()
    if text in {'1', 'true', 'yes', 'y', 'valid', 'pass'}:
        return True
    if text in {'0', 'false', 'no', 'n', 'invalid', 'fail'}:
        return False
    return None


def pick_column(df: pd.DataFrame, candidates: list[str]):
    columns_lower = {c.lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate.lower() in columns_lower:
            return columns_lower[candidate.lower()]
    return None


def resume_id_from_path(path: Path):
    return path.stem


def precision_recall_f1(tp, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) else np.nan
    recall = tp / (tp + fn) if (tp + fn) else np.nan
    if pd.isna(precision) or pd.isna(recall) or (precision + recall) == 0:
        f1 = np.nan
    else:
        f1 = 2 * precision * recall / (precision + recall)
    return precision, recall, f1


def ndcg_at_k(predicted_ids, relevance_map, k=5):
    if not predicted_ids or not relevance_map:
        return np.nan

    dcg = 0.0
    for i, job_id in enumerate(predicted_ids[:k], start=1):
        rel = float(relevance_map.get(job_id, 0.0))
        if rel > 0:
            dcg += (2**rel - 1) / math.log2(i + 1)

    ideal_rels = sorted([float(v) for v in relevance_map.values() if float(v) > 0], reverse=True)[:k]
    if not ideal_rels:
        return np.nan

    idcg = 0.0
    for i, rel in enumerate(ideal_rels, start=1):
        idcg += (2**rel - 1) / math.log2(i + 1)

    return (dcg / idcg) if idcg > 0 else np.nan


## 8) Skill Extraction Evaluation

For each sample resume:
1. Parse and analyze with current backend modules.
2. Extract detected skill IDs.
3. Compare against `expected_skills.csv`.
4. Compute Precision, Recall, and F1-score.


In [ ]:
skill_metrics_rows = []
skill_tp = skill_fp = skill_fn = 0

resume_col_skills = pick_column(expected_skills_df, ['resume_id', 'resume', 'resume_name', 'file_name', 'filename'])
skill_col = pick_column(expected_skills_df, ['skill_id', 'expected_skill_id'])

if not backend_ready:
    warn('Skipping skill extraction evaluation because backend imports failed.')
elif not resume_files:
    warn('Skipping skill extraction evaluation because no sample resumes were found.')
elif expected_skills_df.empty or resume_col_skills is None or skill_col is None:
    warn('Skipping skill extraction metrics because expected_skills.csv is empty or missing required columns.')
else:
    skill_extractor = SkillExtractor()

    expected_by_resume = defaultdict(set)
    for _, row in expected_skills_df.iterrows():
        rid = normalize_text(row.get(resume_col_skills, ''))
        sid = str(row.get(skill_col, '')).strip().upper()
        if rid and sid:
            expected_by_resume[rid].add(sid)

    for resume_path in resume_files:
        rid = resume_id_from_path(resume_path)
        rid_key = normalize_text(rid)

        try:
            analysis = analyze_resume_file(str(resume_path), skill_extractor)
            resume_analysis_cache[rid] = analysis
            predicted_ids = {
                str(item.get('skill_id', '')).strip().upper()
                for item in analysis.get('candidate_profile', {}).get('skills', [])
                if str(item.get('skill_id', '')).strip()
            }
        except Exception as exc:
            warn(f'Skill extraction failed for {resume_path.name}: {exc}')
            predicted_ids = set()

        expected_ids = expected_by_resume.get(rid_key, set())

        if not expected_ids:
            continue

        tp = len(predicted_ids & expected_ids)
        fp = len(predicted_ids - expected_ids)
        fn = len(expected_ids - predicted_ids)

        p, r, f1 = precision_recall_f1(tp, fp, fn)
        skill_metrics_rows.append({
            'resume_id': rid,
            'tp': tp,
            'fp': fp,
            'fn': fn,
            'precision': p,
            'recall': r,
            'f1': f1,
            'predicted_skill_count': len(predicted_ids),
            'expected_skill_count': len(expected_ids),
        })

        skill_tp += tp
        skill_fp += fp
        skill_fn += fn

skill_precision, skill_recall, skill_f1 = precision_recall_f1(skill_tp, skill_fp, skill_fn)
skill_metrics_df = pd.DataFrame(skill_metrics_rows)

print('Skill Extraction Summary')
print('TP =', skill_tp, 'FP =', skill_fp, 'FN =', skill_fn)
print('Precision =', skill_precision)
print('Recall =', skill_recall)
print('F1 =', skill_f1)

if not skill_metrics_df.empty:
    display(skill_metrics_df.head(20))


## 9) Job Matching Evaluation

For each sample resume:
1. Run current baseline job matcher.
2. Compare ranked jobs with `expected_job_matches.csv`.
3. Compute Precision@3, Precision@5, NDCG@5, and MRR.


In [ ]:
job_metrics_rows = []

job_p3_list = []
job_p5_list = []
job_ndcg5_list = []
job_mrr_list = []

resume_col_jobs = pick_column(expected_jobs_df, ['resume_id', 'resume', 'resume_name', 'file_name', 'filename'])
job_id_col = pick_column(expected_jobs_df, ['job_id', 'expected_job_id'])
rel_col = pick_column(expected_jobs_df, ['relevance', 'gain', 'score'])
rank_col = pick_column(expected_jobs_df, ['expected_rank', 'rank'])

if not backend_ready:
    warn('Skipping job matching evaluation because backend imports failed.')
elif not resume_files:
    warn('Skipping job matching evaluation because no sample resumes were found.')
elif expected_jobs_df.empty or resume_col_jobs is None or job_id_col is None:
    warn('Skipping job matching metrics because expected_job_matches.csv is empty or missing required columns.')
else:
    matcher = JobMatcher()

    expected_jobs_grouped = defaultdict(list)
    for _, row in expected_jobs_df.iterrows():
        rid = normalize_text(row.get(resume_col_jobs, ''))
        jid = str(row.get(job_id_col, '')).strip()
        if rid and jid:
            if rel_col:
                try:
                    rel_val = float(str(row.get(rel_col, '1')).strip() or 1)
                except Exception:
                    rel_val = 1.0
            elif rank_col:
                try:
                    rank_val = int(float(str(row.get(rank_col, '1')).strip() or 1))
                    rel_val = max(1.0, float(6 - min(rank_val, 5)))
                except Exception:
                    rel_val = 1.0
            else:
                rel_val = 1.0
            expected_jobs_grouped[rid].append((jid, rel_val))

    for resume_path in resume_files:
        rid = resume_id_from_path(resume_path)
        rid_key = normalize_text(rid)
        expected_pairs = expected_jobs_grouped.get(rid_key, [])

        if not expected_pairs:
            continue

        if rid not in resume_analysis_cache:
            try:
                analyzer = analyze_resume_file(str(resume_path), SkillExtractor())
                resume_analysis_cache[rid] = analyzer
            except Exception as exc:
                warn(f'Could not analyze {resume_path.name} for job matching: {exc}')
                continue

        analysis = resume_analysis_cache[rid]
        resume_text = analysis.get('normalized_for_matching', {}).get('resume_text_for_matching', '')
        candidate_skill_ids = analysis.get('normalized_for_matching', {}).get('skill_ids', [])

        try:
            match_result = matcher.match_jobs(
                resume_text_for_matching=resume_text,
                candidate_skill_ids=candidate_skill_ids,
                preferences={},
                top_n_retrieval=50,
                top_n_output=10,
            )
            matching_cache[rid] = match_result
        except Exception as exc:
            warn(f'Job matching failed for {resume_path.name}: {exc}')
            continue

        predicted_ids = [
            str(item.get('job_id', '')).strip()
            for item in match_result.get('fit_now_matches', [])
            if str(item.get('job_id', '')).strip()
        ]

        expected_ids = {jid for jid, _ in expected_pairs}
        relevance_map = {jid: rel for jid, rel in expected_pairs}

        p3 = len(set(predicted_ids[:3]) & expected_ids) / 3 if len(predicted_ids) >= 3 else np.nan
        p5 = len(set(predicted_ids[:5]) & expected_ids) / 5 if len(predicted_ids) >= 5 else np.nan
        ndcg5 = ndcg_at_k(predicted_ids, relevance_map, k=5)

        rr = 0.0
        for idx, jid in enumerate(predicted_ids, start=1):
            if jid in expected_ids:
                rr = 1.0 / idx
                break

        mrr = rr if expected_ids else np.nan

        job_p3_list.append(p3)
        job_p5_list.append(p5)
        job_ndcg5_list.append(ndcg5)
        job_mrr_list.append(mrr)

        job_metrics_rows.append({
            'resume_id': rid,
            'precision_at_3': p3,
            'precision_at_5': p5,
            'ndcg_at_5': ndcg5,
            'mrr': mrr,
            'expected_jobs_count': len(expected_ids),
            'predicted_jobs_count': len(predicted_ids),
        })

job_metrics_df = pd.DataFrame(job_metrics_rows)

job_precision_at_3 = float(np.nanmean(job_p3_list)) if job_p3_list else np.nan
job_precision_at_5 = float(np.nanmean(job_p5_list)) if job_p5_list else np.nan
job_ndcg_at_5 = float(np.nanmean(job_ndcg5_list)) if job_ndcg5_list else np.nan
job_mrr = float(np.nanmean(job_mrr_list)) if job_mrr_list else np.nan

print('Job Matching Summary')
print('Precision@3 =', job_precision_at_3)
print('Precision@5 =', job_precision_at_5)
print('NDCG@5 =', job_ndcg_at_5)
print('MRR =', job_mrr)

if not job_metrics_df.empty:
    display(job_metrics_df.head(20))


## 10) Skill Gap Evaluation

For expected resume-job pairs:
- Compare generated missing skills against `expected_skill_gaps.csv`.
- Compute Gap Precision, Gap Recall, and Gap F1-score.


In [ ]:
gap_metrics_rows = []

gap_tp = gap_fp = gap_fn = 0

resume_col_gaps = pick_column(expected_gaps_df, ['resume_id', 'resume', 'resume_name', 'file_name', 'filename'])
job_col_gaps = pick_column(expected_gaps_df, ['job_id', 'expected_job_id'])
skill_col_gaps = pick_column(expected_gaps_df, ['skill_id', 'missing_skill_id', 'gap_skill_id'])

if expected_gaps_df.empty or resume_col_gaps is None or skill_col_gaps is None:
    warn('Skipping skill gap metrics because expected_skill_gaps.csv is empty or missing required columns.')
else:
    expected_groups = defaultdict(set)
    for _, row in expected_gaps_df.iterrows():
        rid = normalize_text(row.get(resume_col_gaps, ''))
        jid = normalize_text(row.get(job_col_gaps, '')) if job_col_gaps else ''
        sid = str(row.get(skill_col_gaps, '')).strip().upper()
        if rid and sid:
            expected_groups[(rid, jid)].add(sid)

    for (rid_key, jid_key), expected_set in expected_groups.items():
        rid = rid_key
        if rid not in matching_cache:
            # Try to compute matcher output from cached analysis if available.
            if rid in resume_analysis_cache and backend_ready:
                try:
                    matcher = JobMatcher()
                    analysis = resume_analysis_cache[rid]
                    matching_cache[rid] = matcher.match_jobs(
                        resume_text_for_matching=analysis.get('normalized_for_matching', {}).get('resume_text_for_matching', ''),
                        candidate_skill_ids=analysis.get('normalized_for_matching', {}).get('skill_ids', []),
                        preferences={},
                        top_n_retrieval=50,
                        top_n_output=10,
                    )
                except Exception as exc:
                    warn(f'Could not run matcher for gap eval resume_id={rid}: {exc}')
                    continue
            else:
                continue

        match_result = matching_cache.get(rid, {})
        predicted_set = set()

        if jid_key:
            combined_jobs = list(match_result.get('fit_now_matches', [])) + list(match_result.get('aspiration_matches', []))
            target = None
            for item in combined_jobs:
                if normalize_text(item.get('job_id', '')) == jid_key:
                    target = item
                    break
            if target:
                predicted_set = {
                    str(s).strip().upper() for s in target.get('missing_skill_ids', []) if str(s).strip()
                }
        else:
            predicted_set = {
                str(item.get('skill_id', '')).strip().upper()
                for item in match_result.get('skill_gaps', [])
                if str(item.get('skill_id', '')).strip()
            }

        tp = len(predicted_set & expected_set)
        fp = len(predicted_set - expected_set)
        fn = len(expected_set - predicted_set)

        p, r, f1 = precision_recall_f1(tp, fp, fn)

        gap_tp += tp
        gap_fp += fp
        gap_fn += fn

        gap_metrics_rows.append({
            'resume_id': rid,
            'job_id': jid_key,
            'tp': tp,
            'fp': fp,
            'fn': fn,
            'gap_precision': p,
            'gap_recall': r,
            'gap_f1': f1,
            'expected_gap_count': len(expected_set),
            'predicted_gap_count': len(predicted_set),
        })

gap_precision, gap_recall, gap_f1 = precision_recall_f1(gap_tp, gap_fp, gap_fn)
gap_metrics_df = pd.DataFrame(gap_metrics_rows)

print('Skill Gap Summary')
print('TP =', gap_tp, 'FP =', gap_fp, 'FN =', gap_fn)
print('Gap Precision =', gap_precision)
print('Gap Recall =', gap_recall)
print('Gap F1 =', gap_f1)

if not gap_metrics_df.empty:
    display(gap_metrics_df.head(20))


## 11) Course Recommendation Evaluation

For each resume:
- Uses current baseline recommendations.
- Compares against `expected_course_recommendations.csv`.
- Computes Precision@5.


In [ ]:
course_metrics_rows = []
course_p5_values = []

resume_col_courses = pick_column(expected_courses_df, ['resume_id', 'resume', 'resume_name', 'file_name', 'filename'])
resource_col = pick_column(expected_courses_df, ['resource_id', 'course_id', 'expected_resource_id'])

if expected_courses_df.empty or resume_col_courses is None or resource_col is None:
    warn('Skipping course recommendation metrics because expected_course_recommendations.csv is empty or missing required columns.')
else:
    expected_course_by_resume = defaultdict(set)
    for _, row in expected_courses_df.iterrows():
        rid = normalize_text(row.get(resume_col_courses, ''))
        resource_id = str(row.get(resource_col, '')).strip()
        if rid and resource_id:
            expected_course_by_resume[rid].add(resource_id)

    for rid_key, expected_set in expected_course_by_resume.items():
        if rid_key not in matching_cache:
            continue

        result = matching_cache[rid_key]
        predicted_ids = [
            str(item.get('resource_id', '')).strip()
            for item in result.get('learning_recommendations', [])[:5]
            if str(item.get('resource_id', '')).strip()
        ]

        if len(predicted_ids) < 5:
            # still evaluate with denominator 5 as Precision@5 convention
            pass

        p5 = len(set(predicted_ids) & expected_set) / 5
        course_p5_values.append(p5)

        course_metrics_rows.append({
            'resume_id': rid_key,
            'precision_at_5': p5,
            'expected_recommendation_count': len(expected_set),
            'predicted_top5_count': len(predicted_ids),
        })

course_precision_at_5 = float(np.nanmean(course_p5_values)) if course_p5_values else np.nan
course_metrics_df = pd.DataFrame(course_metrics_rows)

print('Course Recommendation Summary')
print('Precision@5 =', course_precision_at_5)

if not course_metrics_df.empty:
    display(course_metrics_df.head(20))


## 12) File Validation Evaluation

Uses `file_validation_cases.csv` to test valid/invalid files where available.
Summarizes accepted, rejected, warnings, and pass rate.


In [ ]:
file_eval_rows = []

validation_file_col = pick_column(file_validation_cases_df, ['file_path', 'file_name', 'filename', 'document'])
validation_expected_col = pick_column(file_validation_cases_df, ['expected_valid', 'is_valid', 'expected_result'])

accepted_count = 0
rejected_count = 0
warning_count = 0
checked_with_expectation = 0
correct_count = 0

if file_validation_cases_df.empty or validation_file_col is None:
    warn('Skipping file validation metrics because file_validation_cases.csv is empty or missing file column.')
else:
    for _, row in file_validation_cases_df.iterrows():
        file_ref = str(row.get(validation_file_col, '')).strip()
        if not file_ref:
            continue

        file_path = Path(file_ref)
        if not file_path.is_absolute():
            # Try sample_resumes, then evaluation folder.
            candidate_1 = SAMPLE_RESUMES_DIR / file_ref
            candidate_2 = EVALUATION_DIR / file_ref
            file_path = candidate_1 if candidate_1.exists() else candidate_2

        if not file_path.exists() or not file_path.is_file():
            warning_count += 1
            warn(f'Validation file not found: {file_ref}')
            file_eval_rows.append({
                'file_name': file_ref,
                'exists': False,
                'predicted_valid': np.nan,
                'expected_valid': row.get(validation_expected_col, '') if validation_expected_col else '',
                'matched_expected': np.nan,
                'message': 'File not found',
            })
            continue

        try:
            file_bytes = file_path.read_bytes()
            predicted_valid, message = validate_certificate_pdf(file_path.name, file_bytes)
        except Exception as exc:
            warning_count += 1
            warn(f'Validation crashed for {file_ref}: {exc}')
            predicted_valid, message = False, f'Validation error: {exc}'

        if predicted_valid:
            accepted_count += 1
        else:
            rejected_count += 1

        expected_bool = to_bool(row.get(validation_expected_col, '')) if validation_expected_col else None
        matched_expected = np.nan
        if expected_bool is not None:
            checked_with_expectation += 1
            matched_expected = (predicted_valid == expected_bool)
            if matched_expected:
                correct_count += 1

        file_eval_rows.append({
            'file_name': file_path.name,
            'exists': True,
            'predicted_valid': predicted_valid,
            'expected_valid': expected_bool,
            'matched_expected': matched_expected,
            'message': message,
        })

file_eval_df = pd.DataFrame(file_eval_rows)
file_validation_pass_rate = (correct_count / checked_with_expectation) if checked_with_expectation else np.nan

print('File Validation Summary')
print('Accepted =', accepted_count)
print('Rejected =', rejected_count)
print('Warnings =', warning_count)
print('Pass Rate =', file_validation_pass_rate)

if not file_eval_df.empty:
    display(file_eval_df.head(20))


## 13) Save Evaluation Results

Saves:
- `backend/evaluation/outputs/evaluation_results.csv`
- `backend/evaluation/outputs/evaluation_summary.json`


In [ ]:
from datetime import datetime

summary = {
    'timestamp_utc': datetime.utcnow().isoformat() + 'Z',
    'skill_extraction_precision': None if pd.isna(skill_precision) else float(skill_precision),
    'skill_extraction_recall': None if pd.isna(skill_recall) else float(skill_recall),
    'skill_extraction_f1': None if pd.isna(skill_f1) else float(skill_f1),
    'job_matching_precision_at_3': None if pd.isna(job_precision_at_3) else float(job_precision_at_3),
    'job_matching_precision_at_5': None if pd.isna(job_precision_at_5) else float(job_precision_at_5),
    'job_matching_ndcg_at_5': None if pd.isna(job_ndcg_at_5) else float(job_ndcg_at_5),
    'job_matching_mrr': None if pd.isna(job_mrr) else float(job_mrr),
    'skill_gap_precision': None if pd.isna(gap_precision) else float(gap_precision),
    'skill_gap_recall': None if pd.isna(gap_recall) else float(gap_recall),
    'skill_gap_f1': None if pd.isna(gap_f1) else float(gap_f1),
    'course_recommendation_precision_at_5': None if pd.isna(course_precision_at_5) else float(course_precision_at_5),
    'file_validation_pass_rate': None if pd.isna(file_validation_pass_rate) else float(file_validation_pass_rate),
    'warnings_count': len(warnings_log),
}

result_rows = [
    {'metric': 'Skill Extraction Precision', 'value': summary['skill_extraction_precision']},
    {'metric': 'Skill Extraction Recall', 'value': summary['skill_extraction_recall']},
    {'metric': 'Skill Extraction F1', 'value': summary['skill_extraction_f1']},
    {'metric': 'Job Matching Precision@3', 'value': summary['job_matching_precision_at_3']},
    {'metric': 'Job Matching Precision@5', 'value': summary['job_matching_precision_at_5']},
    {'metric': 'Job Matching NDCG@5', 'value': summary['job_matching_ndcg_at_5']},
    {'metric': 'Job Matching MRR', 'value': summary['job_matching_mrr']},
    {'metric': 'Skill Gap Precision', 'value': summary['skill_gap_precision']},
    {'metric': 'Skill Gap Recall', 'value': summary['skill_gap_recall']},
    {'metric': 'Skill Gap F1', 'value': summary['skill_gap_f1']},
    {'metric': 'Course Recommendation Precision@5', 'value': summary['course_recommendation_precision_at_5']},
    {'metric': 'File Validation Pass Rate', 'value': summary['file_validation_pass_rate']},
]

results_df = pd.DataFrame(result_rows)

results_csv_path = OUTPUTS_DIR / 'evaluation_results.csv'
summary_json_path = OUTPUTS_DIR / 'evaluation_summary.json'

results_df.to_csv(results_csv_path, index=False)
with open(summary_json_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

print('Saved:', results_csv_path)
print('Saved:', summary_json_path)


## 14) Thesis Summary Output

The table below represents **baseline results only** (before any improved competency-aware model).


In [ ]:
summary_table = pd.DataFrame([
    {'Metric': 'Skill Extraction Precision', 'Value': summary.get('skill_extraction_precision')},
    {'Metric': 'Skill Extraction Recall', 'Value': summary.get('skill_extraction_recall')},
    {'Metric': 'Skill Extraction F1', 'Value': summary.get('skill_extraction_f1')},
    {'Metric': 'Job Matching Precision@3', 'Value': summary.get('job_matching_precision_at_3')},
    {'Metric': 'Job Matching Precision@5', 'Value': summary.get('job_matching_precision_at_5')},
    {'Metric': 'Job Matching NDCG@5', 'Value': summary.get('job_matching_ndcg_at_5')},
    {'Metric': 'Skill Gap Precision', 'Value': summary.get('skill_gap_precision')},
    {'Metric': 'Skill Gap Recall', 'Value': summary.get('skill_gap_recall')},
    {'Metric': 'Skill Gap F1', 'Value': summary.get('skill_gap_f1')},
    {'Metric': 'Course Recommendation Precision@5', 'Value': summary.get('course_recommendation_precision_at_5')},
    {'Metric': 'File Validation Pass Rate', 'Value': summary.get('file_validation_pass_rate')},
])

print('Kareerly Baseline Evaluation Summary')
display(summary_table)

if warnings_log:
    print('
Warnings captured during run:')
    for idx, msg in enumerate(warnings_log[:50], start=1):
        print(f'{idx}. {msg}')


### Baseline Interpretation Note

These values are the benchmark for the **current baseline hybrid model**.
Use this as the reference point before implementing and comparing the planned competency relationship–aware upgrade.
